# Libraries and Modules

In [16]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import confusion_matrix, classification_report

# Load Dataset

In [32]:
data = pd.read_csv('/content/drive/MyDrive/DS4SE/mini-project-1/data.csv', sep=',')

print("Dataset Sample:", data.head())

Dataset Sample:    Unnamed: 0  LOC  Complexity  Dev_Experience Module_Type  Dev_Mood  \
0           0  152           9               9          UI         1   
1           1  485          10               1          UI         4   
2           2  320          30               5         API         6   
3           3  156           8               4         API         7   
4           4  121          23               8     Backend         3   

   Commit_Message_Length  Buggy  
0                     75      0  
1                     83      0  
2                     38      0  
3                     50      0  
4                     72      0  


# Checking the Correlation (for feature elimination step later)

In [22]:
print("Correlation with target (numerical only):")
print(data.corr(numeric_only=True)['Buggy'].sort_values(ascending=False))

Correlation with target (numerical only):
Buggy                    1.000000
LOC                      0.471411
Complexity               0.294531
Commit_Message_Length    0.140886
Dev_Mood                 0.006689
Dev_Experience          -0.024444
Name: Buggy, dtype: float64


# Feature / Target Split

In [23]:
X = data.drop('Buggy', axis=1)
y = data['Buggy']

# Preprocessing (One-Hot Encoding)

In [24]:
categorical_features = ['Module_Type']
numerical_features = ['LOC', 'Complexity', 'Dev_Experience']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(), categorical_features),
        ('num', 'passthrough', numerical_features)
    ]
)

#  Train-Test Split (80:20, Stratified)

In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("\nTrain size:", X_train.shape)
print("Test size:", X_test.shape)


Train size: (160, 6)
Test size: (40, 6)


In [26]:
# # Fit the preprocessor separately
# X_train_transformed = preprocessor.fit_transform(X_train)

# # Convert to DataFrame for readability
# # Get feature names
# cat_features = preprocessor.named_transformers_['cat'].get_feature_names_out(['Module_Type'])
# all_features = list(cat_features) + numerical_features

# X_train_transformed_df = pd.DataFrame(X_train_transformed, columns=all_features)

# print(X_train_transformed_df.head())

# Create Pipeline

In [27]:
pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', MultinomialNB())
])

# Train Model


In [28]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['Module_Type']),
                                                 ('num', 'passthrough',
                                                  ['LOC', 'Complexity',
                                                   'Dev_Experience'])])),
                ('classifier', MultinomialNB())])

# Predictions

In [29]:
y_pred = pipeline.predict(X_test)

# Evaluation Metrics

In [30]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Confusion Matrix:
[[16 10]
 [ 6  8]]

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.62      0.67        26
           1       0.44      0.57      0.50        14

    accuracy                           0.60        40
   macro avg       0.59      0.59      0.58        40
weighted avg       0.63      0.60      0.61        40



# Stratified $k$-Fold Cross Validation


In [34]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

cv_scores = cross_val_score(pipeline, X, y, cv=skf, scoring='accuracy')

print("\nCross-Validation Scores:", cv_scores)
print("Mean Accuracy:", np.mean(cv_scores))


Cross-Validation Scores: [0.4  0.7  0.6  0.6  0.6  0.5  0.55 0.6  0.85 0.55]
Mean Accuracy: 0.595
